# Imports

In [1]:
import os
import sys
from datetime import datetime
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse

In [2]:
from drvi.utils.misc import hvg_batch

/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Config

In [3]:
dataset_filenames = [
    "~/data/cth_datasets/Blood.h5ad", 
    "~/data/cth_datasets/Bone_marrow.h5ad", 
    "~/data/cth_datasets/Heart.h5ad", 
    "~/data/cth_datasets/Hippocampus.h5ad", 
    "~/data/cth_datasets/Intestine.h5ad", 
    "~/data/cth_datasets/Kidney.h5ad", 
    "~/data/cth_datasets/Liver.h5ad", 
    "~/data/cth_datasets/Lung.h5ad", 
    "~/data/cth_datasets/Lymph_node.h5ad", 
    "~/data/cth_datasets/Pancreas.h5ad", 
    "~/data/cth_datasets/Skeletal_muscle.h5ad", 
    "~/data/cth_datasets/Spleen.h5ad",
]

# Utils

In [4]:
def add_empty_vars(adata, new_var_names):
    """
    Adds new variables (genes) to an AnnData object with zero counts 
    across .X and all .layers using ad.concat.
    """
    new_vars_to_add = [v for v in new_var_names if v not in adata.var_names]
    
    if not new_vars_to_add:
        print("All variables already exist in adata.var.")
        return adata
    
    n_obs = adata.n_obs
    n_new = len(new_vars_to_add)

    dummy = ad.AnnData(
        X=sparse.csr_matrix((n_obs, n_new)),
        var=pd.DataFrame(index=new_vars_to_add),
        obs=adata.obs[[]]
    )

    for layer_name in adata.layers:
        dummy.layers[layer_name] = sparse.csr_matrix((n_obs, n_new))

    new_adata = ad.concat(
        [adata, dummy], 
        axis=1, 
        join="outer", 
        merge="unique", 
        uns_merge="first"
    )
    new_adata.obsm = adata.obsm.copy()
    new_adata.obsp = adata.obsp.copy()
    return new_adata

# Processing

## HVG selection

In [5]:
hvgs = {}

for filename in dataset_filenames:
    filename = Path(filename).expanduser()
    hvg_filename = filename.parent / f"{filename.stem}_hvg4000{filename.suffix}"
    print(filename)
    print(hvg_filename)
    if not hvg_filename.exists():
        adata = sc.read_h5ad(filename)
        print(adata)
        adata.layers["counts"] = adata.raw.X.copy()
        del adata.raw
        ###
        hvg_genes = hvg_batch(adata, batch_key="Dataset", target_genes=4000, adataOut=False)
        hvgs[filename.name] = hvg_genes
        adata_hvg = adata[:, hvg_genes].copy()
        adata_hvg.write_h5ad(hvg_filename)
        print(adata_hvg)
    else:
        adata_hvg = sc.read_h5ad(hvg_filename, backed='r')
        print("N cells:", adata_hvg.n_obs)
        print("N cell types:", adata_hvg.obs['Curated_annotation'].nunique())
        print("N datasets:", adata_hvg.obs['Dataset'].nunique())
        print("N samples:", adata_hvg.obs['donor_id'].nunique())
        print(adata_hvg)
        hvgs[filename.name] = adata_hvg.var.index

/home/icb/amirali.moinfar/data/cth_datasets/Blood.h5ad
/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000.h5ad
N cells: 335916
N cell types: 27
N datasets: 4
N samples: 65
AnnData object with n_obs × n_vars = 335916 × 4000 backed at '/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000.h5ad'
    obs: 'Dataset', 'donor_id', 'development_stage', 'sex', 'suspension_type', 'assay', 'Original_annotation', 'CellHint_harmonised_group', 'cell_type', 'Curated_annotation', 'organism', 'disease', 'tissue'
    var: 'exist_in_Ren2021', 'exist_in_DominguezConde2022', 'exist_in_Stephenson2021', 'exist_in_Yoshida2021'
    uns: 'schema_version', 'title'
    obsm: 'X_umap'
    layers: 'counts'
/home/icb/amirali.moinfar/data/cth_datasets/Bone_marrow.h5ad
/home/icb/amirali.moinfar/data/cth_datasets/Bone_marrow_hvg4000.h5ad
N cells: 66613
N cell types: 45
N datasets: 4
N samples: 17
AnnData object with n_obs × n_vars = 66613 × 4000 backed at '/home/icb/amirali.moinfar/data/cth_datasets/Bone_mar

## Splitting blood

In [8]:
filename = dataset_filenames[0]
filename = Path(filename).expanduser()
hvg_filename = filename.parent / f"{filename.stem}_hvg4000{filename.suffix}"
print(hvg_filename)

adata_hvg = sc.read_h5ad(hvg_filename, backed='r')

for ds_name, obs_groups in adata_hvg.obs.groupby("Dataset"):
    print(ds_name, ds_name.split(" ")[0])
    split_filename = hvg_filename.parent / f"{hvg_filename.stem}_{ds_name.split(" ")[0]}{hvg_filename.suffix}"
    print(split_filename)

    if split_filename.exists():
        continue

    adata_subset = adata_hvg[obs_groups.index]
    adata_subset.write_h5ad(split_filename)

/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000.h5ad


/tmp/ipykernel_3557171/881685949.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for ds_name, obs_groups in adata_hvg.obs.groupby("Dataset"):


Dominguez Conde et al. 2022 Dominguez
/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000_Dominguez.h5ad
Ren et al. 2021 Ren
/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000_Ren.h5ad
Stephenson et al. 2021 Stephenson
/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000_Stephenson.h5ad
Yoshida et al. 2021 Yoshida
/home/icb/amirali.moinfar/data/cth_datasets/Blood_hvg4000_Yoshida.h5ad
